In [1]:
# using namespace std;

# [1]
from pyspark.sql import SparkSession

# [2]
from pyspark.sql.functions import col, desc, sum, avg

# [3]
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()

# [4]
df_with_col = spark.range(0, 1000000).withColumn("value",col("id")%1000)

# [5]
df_with_col.take(10)

# [6]
print("before partition",df_with_col.rdd.getNumPartitions())

# [7]
df_repartitioned = df_with_col.repartition(12)

# [8]
print("after partition",df_repartitioned.rdd.getNumPartitions())

# [9]
df_repartitioned.write.mode("overwrite").csv("output/employees.csv",header=True)

# [10]
df_colapsed = df_repartitioned.coalesce(2)

# [11]
print("after colapsed",df_colapsed.rdd.getNumPartitions())

# [12]
df_colapsed.write.mode("overwrite").csv("output/employees.csv",header=True)

# [13]
optimized_df = df_repartitioned.filter(col("value") > 500).filter(col("id") < 50000).select("id","value")

# [14]
optimized_df.explain()

# [15]
import time

# [16]
start_time = time.time()
count_uncached = optimized_df.count()
end_time = time.time()
print(f"1. optimized Execution | count : {count_uncached} | time : {round(end_time)}")

# [17]
optimized_df.cache()

# [18]
start_time = time.time()
count_uncached = optimized_df.count()
end_time = time.time()
print(f"1. optimized Execution | count : {count_uncached} | time : {round(end_time)}")

# [ ]
optimized_df.unpersist()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 06:54:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/13 06:54:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

before partition 2


[Stage 1:>                                                          (0 + 2) / 2]

after partition 12


26/06/13 06:54:22 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 5:===========================================================(2 + 0) / 2]

after colapsed 2


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=172]
   +- Project [id#0L, (id#0L % 1000) AS value#2L]
      +- Filter (((id#0L % 1000) > 500) AND (id#0L < 50000))
         +- Range (0, 1000000, step=1, splits=2)


1. optimized Execution | count : 24950 | time : 1781333670
1. optimized Execution | count : 24950 | time : 1781333671


DataFrame[id: bigint, value: bigint]